# Step 4: Funnel Metrics
### Marketing Funnel Analysis: Bank Marketing Campaign Dataset

### Our Funnel Structure
```
Total Contacts
      ↓
Previously Contacted (had prior campaigns)
      ↓
Engaged (duration > 0 seconds)
      ↓
Converted (subscribed = yes)
```

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/bank_cleaned.csv')
print(f"Cleaned dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Cleaned dataset loaded: 41,188 rows × 23 columns


In [12]:
# Stage 1: Total contacts
total_contacts = len(df)

# Stage 2: Previously contacted (had prior campaign)
previously_contacted = df[df['previous'] > 0]

# Stage 3: Engaged (call duration > 0 seconds)
engaged = df[df['duration'] > 0]

# Stage 4: Converted
converted = df[df['y_binary'] == 1]

print("=== FUNNEL STAGES ===")
print(f"Stage 1: Total Contacts:        {total_contacts:>6,}")
print(f"Stage 2: Previously Contacted:  {len(previously_contacted):>6,}")
print(f"Stage 3: Engaged (duration>0):  {len(engaged):>6,}")
print(f"Stage 4: Converted:             {len(converted):>6,}")

=== FUNNEL STAGES ===
Stage 1: Total Contacts:        41,188
Stage 2: Previously Contacted:   5,625
Stage 3: Engaged (duration>0):  41,184
Stage 4: Converted:              4,640


In [4]:
print("=== CONVERSION RATES BETWEEN STAGES ===")

s1_to_s2 = len(previously_contacted) / total_contacts * 100
s2_to_s3 = len(engaged[engaged['previous'] > 0]) / len(previously_contacted) * 100
s3_to_s4 = len(converted[converted['duration'] > 0]) / len(engaged) * 100
overall  = len(converted) / total_contacts * 100

print(f"Total → Previously Contacted:  {s1_to_s2:.2f}%")
print(f"Previously Contacted → Engaged: {s2_to_s3:.2f}%")
print(f"Engaged → Converted:            {s3_to_s4:.2f}%")
print(f"Overall Conversion Rate:        {overall:.2f}%")

=== CONVERSION RATES BETWEEN STAGES ===
Total → Previously Contacted:  13.66%
Previously Contacted → Engaged: 100.00%
Engaged → Converted:            11.27%
Overall Conversion Rate:        11.27%


In [5]:
print("=== DROP-OFF RATES AT EACH STAGE ===")

drop_s1_s2 = 100 - s1_to_s2
drop_s2_s3 = 100 - s2_to_s3
drop_s3_s4 = 100 - s3_to_s4

print(f"Drop-off Stage 1→2:  {drop_s1_s2:.2f}% never had prior contact")
print(f"Drop-off Stage 2→3:  {drop_s2_s3:.2f}% didn't engage on call")
print(f"Drop-off Stage 3→4:  {drop_s3_s4:.2f}% engaged but didn't convert")

=== DROP-OFF RATES AT EACH STAGE ===
Drop-off Stage 1→2:  86.34% never had prior contact
Drop-off Stage 2→3:  0.00% didn't engage on call
Drop-off Stage 3→4:  88.73% engaged but didn't convert


In [6]:
print("=== CONVERSION RATE BY CONTACT CHANNEL ===")
channel_conv = df.groupby('contact')['y_binary'].agg(['sum', 'count'])
channel_conv.columns = ['converted', 'total']
channel_conv['conversion_rate_%'] = (channel_conv['converted'] / channel_conv['total'] * 100).round(2)
channel_conv['drop_off_%'] = (100 - channel_conv['conversion_rate_%']).round(2)
print(channel_conv.sort_values('conversion_rate_%', ascending=False))

=== CONVERSION RATE BY CONTACT CHANNEL ===
           converted  total  conversion_rate_%  drop_off_%
contact                                                   
cellular        3853  26144              14.74       85.26
telephone        787  15044               5.23       94.77


In [7]:
print("=== CONVERSION RATE BY CAMPAIGN INTENSITY ===")
campaign_conv = df.groupby('campaign_bucket', observed=True)['y_binary'].agg(['sum', 'count'])
campaign_conv.columns = ['converted', 'total']
campaign_conv['conversion_rate_%'] = (campaign_conv['converted'] / campaign_conv['total'] * 100).round(2)
print(campaign_conv)

=== CONVERSION RATE BY CAMPAIGN INTENSITY ===
                 converted  total  conversion_rate_%
campaign_bucket                                     
1 contact             2300  17642              13.04
2-3 contacts          1785  15911              11.22
4-5 contacts           369   4250               8.68
6+ contacts            186   3385               5.49


In [8]:
print("=== CONVERSION RATE BY AGE GROUP ===")
age_conv = df.groupby('age_group', observed=True)['y_binary'].agg(['sum', 'count'])
age_conv.columns = ['converted', 'total']
age_conv['conversion_rate_%'] = (age_conv['converted'] / age_conv['total'] * 100).round(2)
print(age_conv.sort_values('conversion_rate_%', ascending=False))

=== CONVERSION RATE BY AGE GROUP ===
           converted  total  conversion_rate_%
age_group                                     
65+              290    619              46.85
17-25            349   1666              20.95
56-65            451   2963              15.22
26-35           1740  14847              11.72
46-55            717   8249               8.69
36-45           1093  12844               8.51


In [9]:
print("=== CONVERSION RATE BY MONTH ===")
month_order = ['jan','feb','mar','apr','may','jun',
               'jul','aug','sep','oct','nov','dec']
month_conv = df.groupby('month', observed=True)['y_binary'].agg(['sum','count'])
month_conv.columns = ['converted', 'total']
month_conv['conversion_rate_%'] = (month_conv['converted'] / month_conv['total'] * 100).round(2)
month_conv = month_conv.reindex([m for m in month_order if m in month_conv.index])
print(month_conv)

=== CONVERSION RATE BY MONTH ===
       converted  total  conversion_rate_%
month                                     
mar          276    546              50.55
apr          539   2632              20.48
may          886  13769               6.43
jun          559   5318              10.51
jul          649   7174               9.05
aug          655   6178              10.60
sep          256    570              44.91
oct          315    718              43.87
nov          416   4101              10.14
dec           89    182              48.90


In [10]:
funnel_summary = pd.DataFrame({
    'Stage': [
        '1. Total Contacts',
        '2. Previously Contacted',
        '3. Engaged',
        '4. Converted'
    ],
    'Count': [
        total_contacts,
        len(previously_contacted),
        len(engaged),
        len(converted)
    ]
})

funnel_summary['Drop_from_previous'] = funnel_summary['Count'].diff().abs()
funnel_summary['Conversion_%'] = (funnel_summary['Count'] / total_contacts * 100).round(2)
print("=== FULL FUNNEL SUMMARY TABLE ===")
print(funnel_summary.to_string(index=False))

=== FULL FUNNEL SUMMARY TABLE ===
                  Stage  Count  Drop_from_previous  Conversion_%
      1. Total Contacts  41188                 NaN        100.00
2. Previously Contacted   5625             35563.0         13.66
             3. Engaged  41184             35559.0         99.99
           4. Converted   4640             36544.0         11.27


In [11]:
funnel_summary.to_csv('../outputs/funnel_summary.csv', index=False)
channel_conv.to_csv('../outputs/channel_conversion.csv')
age_conv.to_csv('../outputs/age_conversion.csv')
month_conv.to_csv('../outputs/month_conversion.csv')

print("All funnel metrics saved to outputs/")

All funnel metrics saved to outputs/


## Funnel Metrics Summary
- Biggest drop-off is at Stage 1→2: most contacts had no prior relationship
- Cellular channel outperforms telephone significantly
- Fewer campaign contacts = higher conversion (quality over quantity)
- Certain months show dramatically higher conversion rates
- Age groups show varying conversion patterns